[Reference](https://medium.com/@Rohan_Dutt/how-to-write-custom-loss-functions-that-solve-specific-business-problems-e020615eb746)

In [1]:
import torch
import torch.nn as nn

class AsymmetricStockLoss(nn.Module):
    def __init__(self, over_penalty=3.0, under_penalty=1.0):
        super().__init__()
        self.over_penalty = over_penalty
        self.under_penalty = under_penalty
    def forward(self, y_pred, y_true):
        diff = y_pred - y_true
        over_loss = torch.relu(diff) ** 2 * self.over_penalty
        under_loss = torch.relu(-diff) ** 2 * self.under_penalty
        return (over_loss + under_loss).mean()
# Example use
loss_fn = AsymmetricStockLoss(over_penalty=3.0, under_penalty=1.0)
y_true = torch.tensor([100.0])
y_pred = torch.tensor([130.0])  # overstock
print(loss_fn(y_pred, y_true))

tensor(2700.)


In [3]:
import torch
import torch.nn as nn

class RecencyWeightedMSE(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.weights = weights  # higher weights for recent data
    def forward(self, y_pred, y_true):
        loss = self.weights * (y_pred - y_true) ** 2
        return loss.mean()
# Example
weights = torch.tensor([0.2, 0.5, 1.0])  # oldest to newest
y_true = torch.tensor([120., 150., 180.])
y_pred = torch.tensor([100., 160., 200.])
loss_fn = RecencyWeightedMSE(weights)
print(loss_fn(y_pred, y_true))

tensor(176.6667)


In [5]:
import torch
import torch.nn as nn

class AsymmetricRiskLoss(nn.Module):
    def __init__(self, fn_penalty=5.0, fp_penalty=1.0):
        super().__init__()
        self.fn_penalty = fn_penalty
        self.fp_penalty = fp_penalty
    def forward(self, y_pred, y_true):
        pred = torch.sigmoid(y_pred)
        fn = (1 - pred) * y_true * self.fn_penalty   # missed risky cases
        fp = pred * (1 - y_true) * self.fp_penalty   # over cautious approvals
        return (fn + fp).mean()
# Example
y_true = torch.tensor([1., 0., 1.])
y_pred = torch.tensor([-1.0, 2.0, -0.2])
loss_fn = AsymmetricRiskLoss(fn_penalty=5.0)
print(loss_fn(y_pred, y_true))

tensor(2.4284)


In [6]:
import torch
from torch.autograd import gradcheck, gradgradcheck

def custom_loss_fn(x):
    return (x ** 3).sum()  # intentionally unstable for demo
x = torch.randn(3, dtype=torch.float64, requires_grad=True)
# First order gradient check
print("gradcheck:", gradcheck(custom_loss_fn, (x,), eps=1e-6))
# Second order gradient check
print("gradgradcheck:", gradgradcheck(custom_loss_fn, (x,), eps=1e-6))

gradcheck: True
gradgradcheck: True


In [7]:
activations = {}

def save_hook(name):
    def hook(module, inp, out):
        activations[name] = out.detach()
    return hook
relu = torch.nn.ReLU()
relu.register_forward_hook(save_hook("relu_layer"))
data = torch.randn(10, 10)
out = relu(data)
print("Dead neurons:", (activations["relu_layer"] == 0).sum().item())

Dead neurons: 46


In [8]:
import torch
import torch.nn as nn

class SLAWeightedLoss(nn.Module):
    def __init__(self, penalty_late=5.0):
        super().__init__()
        self.penalty_late = penalty_late
    def forward(self, y_pred, y_true, sla):
        # sla = 1 if on time, 0 if late
        error = (y_pred - y_true).abs()
        weighted = error * torch.where(sla == 1, 1.0, self.penalty_late)
        return weighted.mean()
# Example
y_true = torch.tensor([10., 12., 15.])
y_pred = torch.tensor([9., 20., 13.])
sla = torch.tensor([1, 0, 0])  # two late deliveries
loss_fn = SLAWeightedLoss(penalty_late=5.0)
print(loss_fn(y_pred, y_true, sla))

tensor(17.)


In [9]:
def validate_loss_alignment(kpi_description, loss_formula):
    if len(kpi_description.strip()) == 0:
        return "Invalid: KPI not defined."
    if "cost" not in loss_formula and "penalty" not in loss_formula:
        return "Warning: Loss might not map to business incentives."
    return "OK: Reasonable alignment."

print(validate_loss_alignment("Reduce late deliveries", "penalty * lateness_error"))

OK: Reasonable alignment.
